# Generation behavior check — with vs without step separator

This test verifies (`docs/findings.md`, 2026-06-11): A prompt ending
with the `\n\n` step separator makes the model write the next step;
a prompt missing it looks like a finished message, and the model tends
to emit EOS immediately — the search then records the half-finished
text as a "completed" trajectory.

- **Check 1**: Test how chat templates render a prompt ending with a
  step separator.
  - Expected: passed, except when using the default Llama template —
    the old vllm1 stack (transformers==4.45) fails to render a prompt
    that ends with the step separator `\n\n`.
- **Check 2**: Test whether the strip-and-reappend pattern preserves
  the step separator.
  - Expected: passed.
- **Check 3**: Test how the presence or absence of the step separator
  affects generation behavior.
  - Expected: when the prompt does end with the step separator, the
    LLM is more likely to continue with the next reasoning step.

In [1]:
import sys
from importlib.metadata import version

print("python      ", sys.version.split()[0])
for pkg in ["transformers", "torch", "vllm"]:
    try:
        print(f"{pkg:<12}", version(pkg))
    except Exception:
        print(f"{pkg:<12}", "not installed")

python       3.11.11
transformers 5.5.4
torch        2.10.0
vllm         0.19.1


## Check 1 — Test how chat templates render a prompt ending with a step separator

This tests how two chat templates render a prompt ending with a step
separator `\n\n` with
`tokenizer.apply_chat_template(continue_final_message=True)`:

- **default Llama template**: applies `| trim` to the prompt, so the
  step separator is lost. Note: On some transformers versions this
  surfaces as a `ValueError: substring not found` instead of a trim.
- **SAL's custom template**: identical except it does *not* trim the
  prompt and therefore preserves the step separator.

In [ ]:
from transformers import AutoTokenizer
from sal.config import Config

base_dir = "/groups/chichengz/tnn/datasets"
tokenizer = AutoTokenizer.from_pretrained(f"{base_dir}/Llama3.2-1B-Instruct")
stock_template = tokenizer.chat_template
config = Config()

conv = [
    {"role": "system", "content": "solve it"},
    {"role": "user", "content": "1+1?"},
    {"role": "assistant", "content": "Step one.\n\n"},
]
results = {}
for name, template in [("stock", stock_template),
                       ("sal", config.custom_chat_template)]:
    tokenizer.chat_template = template
    try:
        templated_conv = tokenizer.apply_chat_template(
            conv, continue_final_message=True, tokenize=False
        )
        results[name] = templated_conv.endswith("\n\n")
        print(f"{name:<6} renders OK, separator preserved: {results[name]}")
    except ValueError as e:
        results[name] = None
        print(f"{name:<6} FAILS ValueError: {e}")

stock  renders OK, separator preserved: False
sal    renders OK, separator preserved: True


## Check 2 — Test whether the strip-and-reappend pattern preserves the step separator

Verifies that the strip-and-reappend pattern used in the search code
produces a prompt ending with `\n\n` in this environment:

```python
clean = text.removesuffix("\n\n")
prompt = apply_chat_template(clean, ...)
prompt = prompt + "\n\n"
```

Expected: `True` in both environments.

In [ ]:
from sal.search.utils import build_conv

tokenizer.chat_template = config.custom_chat_template
current_text = "Step one.\n\nStep two.\n\n"

clean = current_text.removesuffix("\n\n")
convs = [build_conv("1+1?", clean, config.system_prompt)]
templated = tokenizer.apply_chat_template(
    convs,
    add_generation_prompt=False,
    continue_final_message=True,
    date_string="Aug 1 2025",
    tokenize=False,
)[0]
prompt = templated + "\n\n"

result = prompt.endswith("\n\n")
prompts = {"strip_and_reappend": result}
print(f"strip_and_reappend prompt ends with separator: {result}")

## Check 3 — Test how the presence or absence of the step separator affects generation behavior

When the prompt does not end with the step separator, the LLM may emit
EOS immediately and stop. When the prompt does end with the step
separator, the LLM is more likely to continue with the next reasoning
step.

In [ ]:
# Optional -- loads Llama3.2-1B on GPU (~2.5 GB in fp16).
import torch
from transformers import AutoModelForCausalLM

EOS_IDS = [128001, 128008, 128009]  # llama-3 end-of-text / eom / eot

# Real prm800k level-4 question + the first two steps the model
# generated for it in a recorded run.
question = (
    "The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]is a line."
    "\n\nThe set of points $(x,y,z)$ that satisfy\n\\[6x = -y = -4z\\]is "
    "another line.\n\nFind the angle between these lines, in degrees."
)
current_text = (
    "## Step 1: Identify the direction vectors of the lines.\n"
    "The direction vectors of the lines can be found from the coefficients "
    "of x, y, and z in the given equations. For the first line, the "
    "direction vector is (2, 3, -1), and for the second line, the "
    "direction vector is (6, -1, -4).\n\n"
    " We can use these direction vectors to find the angle between the "
    "lines.\n\n"
)

model = AutoModelForCausalLM.from_pretrained(
    f"{base_dir}/Llama3.2-1B-Instruct", torch_dtype=torch.float16
).cuda().eval()

tokenizer.chat_template = config.custom_chat_template
clean = current_text.removesuffix("\n\n")
convs = [build_conv(question, clean, config.system_prompt)]
templated = tokenizer.apply_chat_template(
    convs,
    add_generation_prompt=False,
    continue_final_message=True,
    date_string="Aug 1 2025",
    tokenize=False,
)[0]

p_eos = {}
for name, prompt in [("with_sep", templated + "\n\n"), ("no_sep", templated)]:
    ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.cuda()
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    probs = torch.softmax(logits.float(), dim=-1)
    p_eos[name] = sum(probs[i].item() for i in EOS_IDS)
    print(f"P(first token is EOS | {name:<8}) = {p_eos[name]:.2e}")

ratio = p_eos["no_sep"] / max(p_eos["with_sep"], 1e-12)
print(f"ratio = {ratio:,.0f}x")
assert ratio > 100, "missing separator should make immediate EOS more likely"
print("\nPASS -- missing separator makes immediate EOS far more likely")

## Summary

Passes iff this env should produce complete trajectories.

In [ ]:
checks = {
    "sal template renders": results["sal"] is not None,
    "strip_and_reappend keeps separator": bool(prompts["strip_and_reappend"]),
}
if "ratio" in globals():  # check 3 was run (GPU)
    checks["missing separator raises P(first token = EOS)"] = ratio > 100

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL':<5} {name}")

assert all(checks.values()), (
    "env will NOT produce complete trajectories -- "
    "see docs/findings.md (2026-06-11)"
)
print("\nALL CHECKS PASSED -- env should produce complete trajectories")